In [ ]:
import mlflow
import mlflow.sklearn
import joblib

In [3]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("bank-churn-experiment")

2026/04/18 15:06:26 INFO mlflow.tracking.fluent: Experiment with name 'bank-churn-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1776481586559, experiment_id='1', last_update_time=1776481586559, lifecycle_stage='active', name='bank-churn-experiment', tags={}, trace_location=None, workspace='default'>

In [4]:
import sys
print(sys.executable)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

d:\COURSE\Coursa\ML\Operationalizing ML Models MLOps for Scalable AI\5-github-project\bank-churn-mlops\.venv\Scripts\python.exe


In [5]:
#load the data
df = pd.read_csv("../data/raw/bank_churn_train.csv")
df.head()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
#prepare X and y
X = df.drop(columns=["Exited"])
y = df["Exited"]

In [7]:
#converts features for model training with one-hot encoding
X = pd.get_dummies(X, drop_first=True)
X.head()

,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Surname_Abbie,...,Surname_Zotova,Surname_Zox,Surname_Zubarev,Surname_Zubareva,Surname_Zuev,Surname_Zuyev,Surname_Zuyeva,Geography_Germany,Geography_Spain,Gender_Male
0,15634602,619,42,2,0.00,1,1,1,101348.88,False,...,False,False,False,False,False,False,False,False,False,False
1,15647311,608,41,1,83807.86,1,0,1,112542.58,False,...,False,False,False,False,False,False,False,False,True,False
2,15619304,502,42,8,159660.80,3,1,0,113931.57,False,...,False,False,False,False,False,False,False,False,False,False
3,15701354,699,39,1,0.00,2,0,0,93826.63,False,...,False,False,False,False,False,False,False,False,False,False
4,15737888,850,43,2,125510.82,1,1,1,79084.10,False,...,False,False,False,False,False,False,False,False,True,False


In [ ]:
#split train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

joblib.dump(rf_model, "../models/final_model.joblib")
joblib.dump(X.columns.tolist(), "../models/training_columns.joblib")

print("Saved model and training columns.")

X_train shape: (8000, 2943)
X_test shape: (2000, 2943)


In [9]:
#Model 1: Train Logistic Regression

with mlflow.start_run(run_name="logistic_regression"):
    log_model = LogisticRegression(max_iter=1000)
    log_model.fit(X_train, y_train)

    y_pred = log_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)

    mlflow.sklearn.log_model(log_model, "logistic_regression_model")

    print("Logistic Regression")
    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))

2026/04/18 15:10:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/18 15:10:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logistic Regression
Accuracy : 0.8005
Precision: 0.4694
Recall   : 0.117
F1 Score : 0.1874
🏃 View run logistic_regression at: http://127.0.0.1:5000/#/experiments/1/runs/9037eca6ff974b4fa4d7d499badfba9f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [ ]:
##Model 2: Train RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="random_forest"):
    rf_model = RandomForestClassifier(random_state=42)
    rf_model.fit(X_train, y_train)

    rf_pred = rf_model.predict(X_test)

    rf_accuracy = accuracy_score(y_test, rf_pred)
    rf_precision = precision_score(y_test, rf_pred)
    rf_recall = recall_score(y_test, rf_pred)
    rf_f1 = f1_score(y_test, rf_pred)

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("accuracy", rf_accuracy)
    mlflow.log_metric("precision", rf_precision)
    mlflow.log_metric("recall", rf_recall)
    mlflow.log_metric("f1", rf_f1)

    mlflow.sklearn.log_model(rf_model, "random_forest_model")

    print("Random Forest")
    print("Accuracy :", round(rf_accuracy, 4))
    print("Precision:", round(rf_precision, 4))
    print("Recall   :", round(rf_recall, 4))
    print("F1 Score :", round(rf_f1, 4))

2026/04/18 15:11:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/18 15:11:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest
Accuracy : 0.8595
Precision: 0.8182
Recall   : 0.3664
F1 Score : 0.5062
🏃 View run random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/52d15c6a2e0d49bc9b5fa58b488dffa5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
